In [12]:
import pandas as pd
import numpy as np
import time as time_lib
import warnings
warnings.filterwarnings("ignore")

In [13]:
def normal1(df):
  n = df.shape[1]
  maxs = [df[i].max() for i in range(n)]
  df_out = df.copy()
  for i in range(n):
    df_out[i] = (df[i]-1)/(maxs[i]-1)
  return df_out#, maxs

In [14]:
def normal2(df):
  n = df.shape[1]
  maxs = [df[i].max() for i in range(n)]
  df_out = df.copy()
  for i in range(n):
    df_out[i] = df[i]/maxs[i]
  return df_out#, maxs

In [15]:
def unnormal1(df, maxs):
  m = df.shape[0]
  n = df.shape[1]
  df_out = df.copy()
  for i in range(n):
    df_out[i] = df[i]*maxs[i]
  for j in range(n):
    for i in range(m):
      df_out.iloc[i,j] = round(min(maxs[j],max(1, df_out.iloc[i,j])),0)
  return df_out

In [16]:
def normal3(df):
  import numpy as np
  m = df.shape[0]
  n = df.shape[1]
  maxs = [int(df[i].max()) for i in range(n)]
  probs = {}
  for j in range(n):
    mm = m - df.isna().sum(axis=0)[j]
    probs[j] = [np.count_nonzero(df[j] == k)/mm for k in range(1,maxs[j]+1)]
  df_out = df.copy()
  for j in range(n):
    if maxs[j] == 2:
      df_out[j] = df[j]-1
    else:
      c = maxs[j]
      for i in range(m):
        r = df.iloc[i,j]
        if np.isnan(r):
          df_out.iloc[i,j] = np.nan
        elif r == 1:
          df_out.iloc[i,j] = 0
        elif r == 2:
          df_out.iloc[i,j] = (probs[j][0]+probs[j][1])/(sum(probs[j][:c-1])+sum(probs[j][1:c]))
        else:
          df_out.iloc[i,j] = (sum(probs[j][:int(r)-1])+sum(probs[j][1:int(r)]))/(sum(probs[j][:int(c)-1])+sum(probs[j][1:int(c)]))
  return df_out#, maxs, probs

In [17]:
def unnormal3(df,maxs,probs):
  def T2(x, prob, max):
    y = (2*x-prob[0])/(sum(prob[:max-1])+sum(prob[1:]))
    return y
  import numpy as np
  m = df.shape[0]
  n = df.shape[1]
  df_out = df.copy()
  for j in range(n):
    prob = probs[j]
    max = maxs[j]
    if maxs[j] == 2:
      for i in range(m):
        if df.iloc[i,j] < T2(prob[0],prob,max):
          df_out.iloc[i,j] = 1
        else:
          df_out.iloc[i,j] = 2
    else:
      for i in range(m):
        r = df.iloc[i,j]
        if r < T2(prob[0],prob,max):
          df_out.iloc[i,j] = 1
        elif r > T2(sum(prob[:max-1]),prob,max):
          df_out.iloc[i,j] = max
        for k in range(1,max-1):
          if T2(sum(prob[:k]),prob,max) <= r < T2(sum(prob[:k+1]),prob,max):
            df_out.iloc[i,j] = k+1
  return df_out

In [18]:
# A subroutine that calculates $\tau_b$ coefficient for columns $i$ and $j$ of the input dataframe `df`.

def tau_b(df, i, j):
  import scipy.stats as stats
  dat = df.copy()
  keep_ind = []
  for k in range(len(df)):
    if df.isna().iloc[k,i]+df.isna().iloc[k,j] == 0:
      keep_ind.append(k)
  dat_out = dat.iloc[keep_ind,:]
  tau, p = stats.kendalltau(dat_out.iloc[:,i],dat_out.iloc[:,j])
  return tau

In [19]:
def tau_b_mat(df):
  import numpy as np
  n = df.shape[1]
  tau_mat = np.empty(shape=(n, n), dtype='double')
  for i in range(n):
    for j in range(n):
      tau_mat[i,j] = tau_b(df, i, j)
  return tau_mat

Subroutine: Take a column of a missing data and return the imputed one based on monotonization

In [20]:
# new optimization procedure (2023.04.18)
def monotonization(input):

  import numpy as np
  n = len(input)
  c = np.nanmax(input)
  # k = n - np.isnan(input).sum()
  input_obs = input[~np.isnan(input)]
  obs = len(input_obs)
  
  ind = np.where(1-np.isnan(input))
  K = ind[0]

  import gurobipy as gp
  m = gp.Env(empty=True)
  m.setParam('WLSACCESSID', '54ea4edc-45f3-4ac2-b06f-cc4809a3c06c')
  m.setParam('WLSSECRET', '2802dfc5-13c3-4ceb-b6c2-b217b0d5994a')
  m.setParam('LICENSEID', 947226)
  m.setParam("OutputFlag", 0) #add by CHP
  m.start()

# Create the model within the Gurobi environment
  m = gp.Model(env=m)

  from gurobipy import GRB

  y = m.addVars(obs, lb=1, ub=c, name="y")
  u = m.addVars(obs, lb=0, name="u")
  w = m.addVars(obs, lb=0, name="w")

  m.setObjective(gp.quicksum(u[i]+w[i] for i in range(obs)), GRB.MINIMIZE)

  m.addConstr(1 <= y[0])
  m.addConstr(y[obs-1] <= c)
  m.addConstrs(y[i] <= y[i+1] for i in range(obs-1))
  m.addConstrs(y[j]-input_obs[j] == u[j]-w[j] for j in range(obs))

  m.optimize()

  y_out = m.getAttr('X', y)

  x_out = input

  # print(x_out)
  # print(y_out)
  # print(K)

  # for j in range(n):
  #   if j <= (K[0]+K[1])/2:
  #     x_out[j] = y_out[0]
  #   elif j > (K[obs-2]+K[obs-1])/2:
  #     x_out[j] = y_out[obs-1]
  #   else:
  #     for i in range(1,obs-1):
  #       if (K[i-1]+K[i])/2 < j <= (K[i]+K[i+1])/2:
  #         x_out[j] = y_out[i]
  
  for j in range(n):
    if j <= K[0]:
      x_out[j] = y_out[0]
    for k in range(obs-1):
      if j >= K[k] and j < K[k+1]:
        x_out[j] = y_out[k+1]
      if j >= K[k+1]:
        x_out[j] = y_out[obs-1]
      
  for i in range(obs):
    x_out[K[i]] = input_obs[i]

  return x_out


In [21]:
#main loop 
import time as time_lib

k = 10 # k-fold validation 
n_items_list = [  1000, 1000, 1000, 1000, 1000]
m_users_list = [   50, 100, 300, 500, 700 ]
random_seed = 20230808

#prepare the result summary table 
result_summary_df = pd.DataFrame(columns = ['items','users','sparsity','avg_acc','std_acc','avg_rmse','std_rmse','avg_mad','std_mad','avg_time','std_time'])
# 6개의 열을 가진 빈 DataFrame 생성
result_detail_df = pd.DataFrame(columns=['n_items','m_users','acc', 'rmse', 'mad', 'time'])
#main 

for n, m in zip(n_items_list, m_users_list):
    n_items = n 
    n_users = m 

    ten = [str(j).zfill(2) for j in list(range(1,k+1))]
    
    labs = ['test'+i for i in ten]
    labs.insert(0,'orig')
    
    url_dict = {}
    url_dict["orig"] = './data/'+str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv'
    
    for i in ten:
        url_dict["test{0}".format(i)] = './data/'+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+i+'.csv'
    print(labs)
    
    for lab in labs:
        print(url_dict[lab])    
    
    df_dict = {}
    for lab in labs:
        df = pd.read_csv(url_dict[lab])
        df_dict[lab] = df.set_index('item_id')

    #compute the sparsity 
    n_row = df_dict["orig"].shape[0]
    n_col = df_dict["orig"].shape[1]
    Obs_ind = np.where(df_dict["orig"].notnull())    # Row and column indices for the observed entries of "Mdat"
    num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
    sparsity = 1 - num_Obs / (n_row * n_col)
    print("sparsity: "+str(sparsity))

    #Run algorithm 
    acc_list = []
    rmse_list = []
    mad_list = []
    time_list = []

    df_orig = df_dict["orig"]
    df_orig.columns = range(df_orig.shape[1])

    mm = df_orig.shape[0]
    nn = df_orig.shape[1]

    labs_test = labs[1:]

    
    count = 0 

    for lab in labs_test:
        count += 1
        
        df = df_dict[lab]
        
        masked_df = pd.DataFrame(np.ma.masked_array(df, np.isnan(df)))
        #df_norm = normal1(masked_df)
        df_norm = normal2(masked_df)
        #df, maxs, probs = normal3(masked_df) #old version
        #df_norm = normal3(masked_df)

        start = time_lib.time()

        row_means = df_norm.mean(axis=1,skipna=True) #NaN 값을 무시하고 데이터프레임의 각 행의 평균구하기 
        df_agg_mat = pd.concat([row_means]*df_norm.shape[1], axis=1)
        df_agg_mat.index = df.index
        df_agg = pd.concat([df, df_agg_mat], axis=1)
        id_column = pd.DataFrame(range(mm))
        id_column.index = df_agg.index
        df_agg = pd.concat([id_column, df_agg], axis=1)
        df_agg.columns = range(2*nn+1)
        
        

        for i in range(nn):
            df_agg_sorted = df_agg.sort_values(by=[df_agg.columns[nn+i+1],df_agg.columns[i+1]])
            index_i = df_agg_sorted.index
            vec = np.array(df_agg_sorted.iloc[:,i+1])
            out = monotonization(vec)
            out_list = []
            for i in range(len(out)):
                out_list.append(out[i])

            out_df = pd.DataFrame(out_list, index=index_i)
            df_agg = pd.concat([df_agg_sorted, out_df], axis=1)

        df_agg.columns = range(3*nn+1)
        df_agg_sorted_final = df_agg.sort_values(by=[df_agg.columns[0]])
        imputed = df_agg_sorted_final.iloc[:,2*nn+1:]

       
        end = time_lib.time()
        time = end - start

        df_orig.index = range(mm)
        imputed.index = range(mm)

        df_orig.columns = range(nn)
        imputed.columns = range(nn)

        #save the result data
        # url = './result/'

        # if count<10:
        #     filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data0'+str(count)+'_mono_imputed.csv'
        # else:
        #     filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+str(count)+'_mono_imputed.csv'
        
        # imputed.to_csv(filename)
        # print("saved the result file as "+str(filename))

        diff = df_orig - imputed
        sq_diff = diff ** 2
        abs_diff = abs(diff)

        n_match = 0
        for i in range(mm):
            for j in range(nn):
                if df_orig.isna().iloc[i,j]+df.isna().iloc[i,j]==1 and df_orig.iloc[i,j] == imputed.iloc[i,j]:
                    n_match += 1
        acc = n_match/(df.isna().sum().sum()-df_orig.isna().sum().sum())
        rmse = (sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) ** 0.5
        mad = abs_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())
        
        acc_list.append(acc)
        rmse_list.append(rmse)
        mad_list.append(mad)
        time_list.append(time)
        df = pd.DataFrame(data=[ acc_list, rmse_list,mad_list,time_list])
        #print(df)
        

    df = pd.DataFrame(data=[ acc_list, rmse_list,mad_list,time_list])
    display("n: "+str(n_items)+" ,m: "+ str(n_users))
    display(df.T)
    df_temp = df.T
    df_temp_2 = df_temp.copy()
    a = [n] * k 
    b = [m] * k
    df_temp_2.rename(columns={df_temp_2.columns[0]: 'acc', df_temp_2.columns[1]: 'rmse', df_temp_2.columns[2]: 'mad',df_temp_2.columns[3]: 'time'}, inplace=True)
    df_temp_2.insert(0,'m_users',b)
    df_temp_2.insert(0,'n_items',a) 

    # result_detail_df와 df_temp을 행 방향으로 이어 붙임
    result_detail_df = pd.concat([result_detail_df, df_temp_2], axis=0)

    #filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data_all_missforest_imputed.csv'
    #df_temp.to_csv(filename)
    #print('saved '+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+ 'the summary result file as' +str(filename))
    # 열별 평균값 계산
    mean_values = df_temp.mean()
    # 열별 표준편차 계산
    std_values = df_temp.std()
    #new_row = [n,m,sparsity]+mean_values.tolist()
    new_row = [n,m,sparsity,mean_values[0],std_values[0],mean_values[1],std_values[1],mean_values[2],std_values[2],mean_values[3],std_values[3]]
    #display(new_row)
    result_summary_df = result_summary_df.append(pd.Series(new_row, index=result_summary_df.columns), ignore_index=True)
    #filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'missforest_imputed.csv'
    #df_temp.to_csv(filename)
    #print("saved the summary result file as "+str(filename))
    display(result_summary_df)


display(result_summary_df)
display(result_detail_df)
        
result_summary_df.to_csv('result_summary_df_mono_avg_1000_norm2.csv')
result_detail_df.to_csv('result_detail_df_mono_avg_1000_norm2.csv')
        

['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
./data/1000-by-50_original_matrix.csv
./data/20230808_1000-by-50_10_fold_test_data01.csv
./data/20230808_1000-by-50_10_fold_test_data02.csv
./data/20230808_1000-by-50_10_fold_test_data03.csv
./data/20230808_1000-by-50_10_fold_test_data04.csv
./data/20230808_1000-by-50_10_fold_test_data05.csv
./data/20230808_1000-by-50_10_fold_test_data06.csv
./data/20230808_1000-by-50_10_fold_test_data07.csv
./data/20230808_1000-by-50_10_fold_test_data08.csv
./data/20230808_1000-by-50_10_fold_test_data09.csv
./data/20230808_1000-by-50_10_fold_test_data10.csv
sparsity: 0.63338


'n: 1000 ,m: 50'

,0,1,2,3
0,0.446809,0.980998,0.676487,8.101412
1,0.436443,1.023190,0.707583,7.032086
2,0.432624,1.030628,0.708674,7.321240
3,0.426077,1.054696,0.733770,7.170790
4,0.428805,1.031950,0.719040,7.031384
5,0.437534,1.064223,0.727769,7.335789
6,0.446263,0.990131,0.679214,7.002984
7,0.439716,1.020788,0.705947,7.877014
8,0.436443,1.010312,0.698854,7.245811
9,0.436205,1.011655,0.700654,7.633657


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,1000.0,50.0,0.63338,0.436692,0.00663,1.021857,0.025752,0.705799,0.018537,7.375217,0.377635


['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
./data/1000-by-100_original_matrix.csv
./data/20230808_1000-by-100_10_fold_test_data01.csv
./data/20230808_1000-by-100_10_fold_test_data02.csv
./data/20230808_1000-by-100_10_fold_test_data03.csv
./data/20230808_1000-by-100_10_fold_test_data04.csv
./data/20230808_1000-by-100_10_fold_test_data05.csv
./data/20230808_1000-by-100_10_fold_test_data06.csv
./data/20230808_1000-by-100_10_fold_test_data07.csv
./data/20230808_1000-by-100_10_fold_test_data08.csv
./data/20230808_1000-by-100_10_fold_test_data09.csv
./data/20230808_1000-by-100_10_fold_test_data10.csv
sparsity: 0.6867099999999999


...

'n: 1000 ,m: 700'

,0,1,2,3
0,0.438894,1.011588,0.698080,40.474385
1,0.436341,1.009831,0.699190,38.945293
2,0.435564,1.014492,0.701077,39.475609
3,0.440448,1.009666,0.694639,39.260985
4,0.435564,1.007740,0.699190,39.945248
5,0.436119,1.013671,0.700078,39.099501
6,0.433344,1.025375,0.709735,39.722749
7,0.435627,1.022391,0.705438,41.992509
8,0.432075,1.028236,0.712986,39.716016
9,0.441509,1.006858,0.693008,40.634223


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,1000.0,50.0,0.633380,0.436692,0.006630,1.021857,0.025752,0.705799,0.018537,7.375217,0.377635
1,1000.0,100.0,0.686710,0.445849,0.006167,1.002231,0.011996,0.688148,0.010128,12.665711,0.713534
2,1000.0,300.0,0.785123,0.444503,0.004925,0.999589,0.010102,0.687402,0.007463,24.867902,0.947827
3,1000.0,500.0,0.837624,0.439720,0.006580,1.010189,0.006585,0.696581,0.007553,32.801159,3.244725
4,1000.0,700.0,0.871296,0.436549,0.002955,1.014985,0.007640,0.701342,0.006314,39.926652,0.909733


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,1000.0,50.0,0.633380,0.436692,0.006630,1.021857,0.025752,0.705799,0.018537,7.375217,0.377635
1,1000.0,100.0,0.686710,0.445849,0.006167,1.002231,0.011996,0.688148,0.010128,12.665711,0.713534
2,1000.0,300.0,0.785123,0.444503,0.004925,0.999589,0.010102,0.687402,0.007463,24.867902,0.947827
3,1000.0,500.0,0.837624,0.439720,0.006580,1.010189,0.006585,0.696581,0.007553,32.801159,3.244725
4,1000.0,700.0,0.871296,0.436549,0.002955,1.014985,0.007640,0.701342,0.006314,39.926652,0.909733


,n_items,m_users,acc,rmse,mad,time
0,1000,50,0.446809,0.980998,0.676487,8.101412
1,1000,50,0.436443,1.023190,0.707583,7.032086
2,1000,50,0.432624,1.030628,0.708674,7.321240
3,1000,50,0.426077,1.054696,0.733770,7.170790
4,1000,50,0.428805,1.031950,0.719040,7.031384
5,1000,50,0.437534,1.064223,0.727769,7.335789
6,1000,50,0.446263,0.990131,0.679214,7.002984
7,1000,50,0.439716,1.020788,0.705947,7.877014
8,1000,50,0.436443,1.010312,0.698854,7.245811
9,1000,50,0.436205,1.011655,0.700654,7.633657
